# Parsing Milestone 2 trên ổ local Colab

Mỗi thành viên chọn TV của mình, đường dẫn snapshot MS1 và chế độ SMOKE hoặc FULL. Chọn CPU trong Runtime.

Notebook chép dữ liệu từng lô về ổ local, dùng tối đa 32 luồng sao chép, rồi parsing và QA tại local. Sau mỗi 25 ID, checkpoint được đóng thành một gói có checksum và lưu lên Drive. ZIP kết quả cuối chứa các bảng, parsed, validation và checkpoint theo cấu trúc chuẩn.

Giữ snapshot MS1 cố định trong suốt lượt chạy. Nếu chuyển từ bản cũ đang chạy, dừng cell cũ và chờ dừng hẳn trước khi dùng bản này. Giữ đúng MEMBER, MS1_INPUT, FULL và thư mục output cũ để khôi phục checkpoint đã có.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Chọn thành viên và dữ liệu

Chọn `MEMBER` từ TV1 đến TV5, rồi nhập `MS1_INPUT` là thư mục data MS1 của mình. Để trống thì dùng `/content/drive/MyDrive/arxiv_ms1_data/<member>/data`. Nếu dùng thư mục được chia sẻ, thêm shortcut vào My Drive và nhập đúng đường dẫn nhìn thấy trên Colab.

Cell cấu hình chỉ có ba mục:

| Mục | Cần chọn gì? |
| --- | --- |
| `MEMBER` | Thành viên của mình, ví dụ `tv3`. Notebook tự lấy đúng dải ID. |
| `MS1_INPUT` | Đường dẫn thư mục data MS1 có `corpus`, `raw`, `state`. Để trống để dùng đường dẫn mặc định nêu trên. |
| `RUN_MODE` | Lần đầu chọn `SMOKE` để thử tối đa ba bài; thử qua rồi chọn `FULL` để xử lý đủ 5.000 ID được giao. |

Sau khi chọn, bấm nút ▶ bên trái cell để áp dụng rồi chạy các cell tiếp theo từ trên xuống. Khi đổi từ SMOKE sang FULL, chạy lại cell cấu hình, cell xác nhận phạm vi và cell parsing; không cần mount hoặc cài đặt lại. Các thiết lập còn lại dùng mặc định: lưu kết quả riêng theo thành viên, tách SMOKE/FULL và tự resume nếu bị ngắt.

In [ ]:
from pathlib import Path
#@title Cấu hình chạy: chỉ cần chọn ba mục
#@markdown Thành viên: chọn đúng TV của mình; range sẽ được lấy tự động.
MEMBER = 'tv1' #@param ["tv1", "tv2", "tv3", "tv4", "tv5"]
#@markdown Dữ liệu: nhập thư mục data MS1; để trống nếu dùng đường dẫn mặc định.
MS1_INPUT = '' #@param {type:"string"}
#@markdown Chế độ: SMOKE = chạy thử ba bài; FULL = chạy toàn bộ dải ID của mình.
RUN_MODE = 'SMOKE' #@param ["SMOKE", "FULL"]
# Thiết lập mặc định; không cần sửa khi chạy theo hướng dẫn.
OUTPUT_ROOT = '/content/drive/MyDrive/arxiv_ms2_results'
RUN_TAG = 'ms2-members-20261110'
SMOKE_IDS = ''
VERIFY_RESUME = False
# Cùng mã production đã nghiệm thu; không đổi riêng từng người.
CODE_REF = '40103bacb5c34a85baecaa16fc78a856044961e8'
RESUME = True
LOCAL_ROOT = Path('/content/arxiv_ms2_local')


In [ ]:
import subprocess, sys
REPO = Path('/content/arxiv-ms2-integration')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/LeoTkiet/arxiv-data-science.git', str(REPO)], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Checkout has local changes; keep them before changing CODE_REF.')
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_SHA = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO) + '[reporting]'], check=True)
# Add the checked-out source even when this runtime previously installed another checkout.
sys.path.insert(0, str(REPO / 'src'))
print('Commit:', CODE_SHA)

## 2. Xác nhận phạm vi của mình

Notebook đọc phân công từ `configs/ms1_group6_ranges.json` trong cùng checkout. Kiểm tra thành viên, input, output và range in dưới đây trước khi chạy cell parsing.

| Thành viên | Dải ID |
| --- | --- |
| TV1 | 2508.12872–2508.17871 |
| TV2 | 2506.15788–2506.20787 |
| TV3 | 2411.08799–2411.13798 |
| TV4 | 2410.18025–2410.23024 |
| TV5 | 2410.03025–2410.08024 |

In [ ]:
import json, re
from arxiv_pipeline.main import expand_range
from arxiv_pipeline.pipeline import parse

MEMBER_ROWS = {'tv1': 15, 'tv2': 24, 'tv3': 56, 'tv4': 59, 'tv5': 62}
if MEMBER not in MEMBER_ROWS or RUN_MODE not in {'SMOKE', 'FULL'}:
    raise ValueError('Chọn đúng MEMBER và RUN_MODE trong danh sách.')
if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]*', RUN_TAG):
    raise ValueError('RUN_TAG chỉ dùng chữ, số, dấu chấm, gạch dưới hoặc gạch nối.')
assignment_config = json.loads((REPO / 'configs/ms1_group6_ranges.json').read_text())
matches = [r for r in assignment_config['ranges'] if r['source_row'] == MEMBER_ROWS[MEMBER]]
if len(matches) != 1:
    raise ValueError('Không xác định được dải phân công từ configs/ms1_group6_ranges.json.')
assignment = matches[0]
assigned_ids = expand_range(assignment['range'])
if len(assigned_ids) != assignment['count']:
    raise ValueError('Dải phân công không khớp số ID được giao.')
input_root = Path(MS1_INPUT.strip() or f'/content/drive/MyDrive/arxiv_ms1_data/{MEMBER}/data')
OUTPUT_BASE = Path(OUTPUT_ROOT) / RUN_TAG / RUN_MODE.lower()
parse.validate_roots(input_root, OUTPUT_BASE / MEMBER)
if RUN_MODE == 'FULL':
    selected = None
elif SMOKE_IDS.strip():
    selected = SMOKE_IDS.replace(',', ' ').split()
else:
    records, errors, _ = parse._discover(input_root.resolve(), None)
    if errors:
        raise ValueError(f'Discovery chưa đọc được: {errors[:3]}')
    selected = [record['arxiv_id'] for record, _, conflict in records
                if not conflict and record.get('status') == 'success'
                and record['arxiv_id'] in set(assigned_ids)][:3]
if selected is not None and (not selected or not set(selected).issubset(assigned_ids)):
    raise ValueError('Cần ID mẫu hợp lệ thuộc dải của thành viên; kiểm tra discovery hoặc SMOKE_IDS.')
JOBS = [dict(member=MEMBER, input=str(input_root), ids=selected,
             total_assigned=assignment['count'], assignment_range=assignment['range'])]
CHECK_RESUME = RUN_MODE == 'SMOKE' or VERIFY_RESUME
print('Thành viên:', MEMBER, '| chế độ:', RUN_MODE, '| range:', assignment['range'])
print('Số ID sẽ xử lý:', len(assigned_ids) if selected is None else len(selected))
print('Input:', input_root)
print('Output:', OUTPUT_BASE / MEMBER)
print('Commit:', CODE_SHA)

## 3. Chạy parsing trên local và lưu checkpoint theo lô

Cell hiển thị giai đoạn chép dữ liệu, số ID đã lưu kết quả về Drive, thời gian mỗi ID trong lô và số stage dùng cache. Lô mặc định có 25 ID. Nếu runtime bị ngắt, các lô đã ghi receipt trên Drive được khôi phục ở lượt chạy sau. Lô đang xử lý sẽ được thử lại từ checkpoint đã lưu gần nhất.

Ổ local cần đủ chỗ cho source của một lô và output đang tích lũy của lượt chạy. Khi hết dung lượng hoặc gặp lỗi sao chép, xem thông báo rồi chạy tiếp trên runtime có đủ chỗ. Các gói đã lưu được giữ trong thư mục output trên Drive.

In [ ]:
from arxiv_pipeline.pipeline.local_staging import run as run_local

# Thiết lập thực thi mặc định; giao diện chọn thành viên vẫn có ba mục.
COPY_WORKERS = 32
BATCH_SIZE = 25
results = []
for job in JOBS:
    destination = OUTPUT_BASE / job['member']
    try:
        result = run_local(
            job, destination, local_root=LOCAL_ROOT, code_sha=CODE_SHA,
            workers=COPY_WORKERS, batch_size=BATCH_SIZE, check_resume=CHECK_RESUME)
    except Exception:
        print('Lượt chạy bị ngắt. Giữ nguyên output Drive để tiếp tục checkpoint:', destination)
        raise
    results.append(result)
    print(job['member'], {
        k: result[k] for k in [
            'complete', 'attempted_papers', 'paper_statuses',
            'cached_stages', 'total_stages', 'full_assignment_accepted']})
    print('ZIP bàn giao:', destination / 'MS2_RESULTS.zip')

## 4. Bàn giao kết quả

Sau khi chạy xong, gửi `MS2_RESULTS.zip`, `acceptance.json` và `delivery.json` trong output FULL của mình cho TV1/TV5. ZIP chứa đầy đủ thư mục của thành viên, gồm tables, parsed, state, validation, manifest và các số đo. `delivery.json` ghi checksum ZIP và thời gian xuất file cuối.

Các file trong `local-batches/` phục vụ khôi phục sau gián đoạn. Giữ thư mục này khi tiếp tục lượt chạy. Chỉ tải ZIP kết quả khi cell đã thông báo lưu thành công và delivery có trạng thái published.

Thời gian benchmark gồm đọc discovery, chép input, khôi phục cache, parsing/QA local và lưu checkpoint theo lô. Thời gian đóng gói/upload ZIP cuối được ghi riêng trong delivery. Một lượt tiếp tục sẽ có số đo và số stage dùng cache tương ứng lượt đó.

SMOKE và FULL dùng output khác nhau. Để chuyển từ SMOKE sang FULL, đổi RUN_MODE rồi chạy lại cell cấu hình, cell xác nhận phạm vi và cell parsing.

In [ ]:
import json
result_dir = OUTPUT_BASE / MEMBER
receipt = result_dir / 'delivery.json'
if receipt.is_file():
    delivery = json.loads(receipt.read_text())
    print('Trạng thái:', delivery['status'])
    if delivery['status'] == 'published':
        print('ZIP bàn giao:', result_dir / delivery['archive'])
        print('SHA-256:', delivery['sha256'])
        print('Tổng thời gian lượt chạy:', delivery['total_elapsed_seconds'], 'giây')
else:
    print('Chạy cell parsing để tạo kết quả.')